# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tal3at-M/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [6]:
#Finding 1: Search rankings deteriorate at accelerated rates once a page crosses a 180-day staleness threshold.
#- Label Origin: Derived from aggregated historical trend logs comparing consecutive 90-day performance windows.
#- Methodology Question: Does the validation design control for query seasonality? An unsegmented temporal split risks attributing natural post-holiday traffic declines to content staleness.

#Finding 2: Multi-signal heuristic scoring increases decay-recovery rate by identifying at-risk URLs early.
#- Label Origin: Measured through post-refresh impression recovery relative to an unrefreshed control set.
#- Methodology Question: Were candidate URLs partitioned out-of-sample by client domain? If evaluation pages share domain-level sitewide authority shocks, reported precision will suffer from optimistic data leakage.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [7]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold, train_test_split

url = "https://raw.githubusercontent.com/Tal3at-M/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Feature setup
df["ctr_calc"] = df["clicks_90d"] / (df["impressions_90d"] + 1e-5)
df["log_impressions"] = np.log1p(df["impressions_90d"])
df["log_clicks"] = np.log1p(df["clicks_90d"])

features = [
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "content_age_days",
    "log_impressions",
    "log_clicks",
    "ctr_calc",
]
X = df[features].fillna(df[features].median())
y = (df["trend_direction"].str.lower() == "down").astype(int)

# 1. Standard Stratified Split (Before - Week 5)
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
rf_strat = RandomForestClassifier(
    n_estimators=100, max_depth=6, random_state=42, n_jobs=-1
)
rf_strat.fit(X_train_s, y_train_s)
prob_s = rf_strat.predict_proba(X_test_s)[:, 1]
top50_strat_idx = np.argsort(prob_s)[::-1][:50]
p50_strat = y_test_s.iloc[top50_strat_idx].mean()

# 2. Honest Grouped Split by client_id (After - Leakage Proof)
gkf = GroupKFold(n_splits=5)
train_g_idx, test_g_idx = next(gkf.split(X, y, groups=df["client_id"]))

rf_group = RandomForestClassifier(
    n_estimators=100, max_depth=6, random_state=42, n_jobs=-1
)
rf_group.fit(X.iloc[train_g_idx], y.iloc[train_g_idx])
prob_g = rf_group.predict_proba(X.iloc[test_g_idx])[:, 1]
top50_group_idx = np.argsort(prob_g)[::-1][:50]
p50_group = y.iloc[test_g_idx].iloc[top50_group_idx].mean()

audit_table = pd.DataFrame(
    {
        "Split Design": [
            "Random Stratified (Baseline)",
            "Honest Grouped by Client (Audited)",
        ],
        "Precision@50": [f"{p50_strat:.2%}", f"{p50_group:.2%}"],
        "Evaluation Risk": [
            "Potential client overlap",
            "Zero client overlap (Strict generalization)",
        ],
    }
)
display(audit_table)

,Split Design,Precision@50,Evaluation Risk
0,Random Stratified (Baseline),86.00%,Potential client overlap
1,Honest Grouped by Client (Audited),74.00%,Zero client overlap (Strict generalization)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [8]:
# Correlation test against final feature matrix
corrs = X.apply(lambda col: col.corr(y)).abs()
print("--- Final Feature Leakage Audit ---")
for col, c in corrs.items():
  print(f"{col:20s}: Absolute Correlation = {c:.4f}")

# Strict boundary checks
assert all(
    corrs < 0.85
), "Target leakage identified: feature correlation exceeds safe threshold!"
assert not any(
    col in X.columns
    for col in [
        "trend_direction",
        "trend_pct",
        "clicks_last_30d",
        "impressions_last_30d",
    ]
), "Lookahead leakage identified: evaluation window fields present!"
print(
    "\nAudit Result: PASSED. Zero target or temporal lookahead leakage detected."
)

--- Final Feature Leakage Audit ---
impressions_90d     : Absolute Correlation = 0.0182
clicks_90d          : Absolute Correlation = 0.0397
avg_position        : Absolute Correlation = 0.0290
content_age_days    : Absolute Correlation = 0.1639
log_impressions     : Absolute Correlation = 0.1775
log_clicks          : Absolute Correlation = 0.0035
ctr_calc            : Absolute Correlation = 0.0619

Audit Result: PASSED. Zero target or temporal lookahead leakage detected.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [9]:
#- Initial Bold Claim:
#"Our machine learning model accurately predicts which URLs Google will penalize and guarantees traffic recovery after editorial refresh."

#- Safe Rewritten Claim (Observed, Directional, Decision-Support):
#"The trained ensemble model identifies empirical patterns of historical search volume decline, providing directional decision-support to help editorial teams prioritize content refresh candidates under out-of-sample evaluation."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.